<a href="https://colab.research.google.com/github/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-/blob/main/notebooks/shuffle_tests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Shuffle tests: is the signal spatial or temporal?

In `cross_dataset_protocol.ipynb` the time gap carries the signal on Sparkov, while location carries it on TabFormer. To test this directly, we train the **Six + location** model three times on each of these datasets:

| Run | What we change |
|---|---|
| Intact | nothing |
| Location shuffled | merchant locations are shuffled **within each card**; every transaction keeps its time |
| Time gap shuffled | time gaps are shuffled **within each card**; every transaction keeps its merchant location (and its own time of day) |

Shuffling within a card keeps each card's own mix of places and gaps, and only breaks the link between a value and the transaction it belongs to. If a shuffle makes PR-AUC drop, that part carries the signal.

If the signal is temporal on Sparkov and spatial on TabFormer, the time shuffle should hurt Sparkov and the location shuffle should hurt TabFormer.

**What is shuffled in each dataset**

| Dataset | Location shuffle (raw columns moved together) | Rebuilt afterwards |
|---|---|---|
| Sparkov | `merch_lat`, `merch_long` | distance from home, distance from previous transaction, velocity |
| TabFormer | `Merchant City`, `Merchant State` | merchant outside home state, abroad, city-changed proxy |

The time-gap shuffle moves the hours-since-previous-transaction value (`gap_log`) between a card's transactions; velocity is rebuilt from the shuffled gaps. The shuffles are done on the full history, before the split.

**Protocol:** the same as `cross_dataset_protocol.ipynb` (same loaders, features, split, Random Forest Tuned, five seeds, F2 threshold on validation, 500 day-block bootstrap resamples). All three runs are scored on the **same** resampled test days, so we can put a confidence interval on each drop. The intact run uses the same Drive folder and run name as `cross_dataset_protocol.ipynb`, so if that notebook has already trained *Six + location* on this dataset, it is loaded instead of retrained.

We run the notebook once per dataset by changing `DATASET` in cell 1.1. Section 6 combines both datasets into Table 2.

## 1. Environment setup
We type credentials in at runtime, so they are never saved in the notebook. We use the same `QUICK` setting as in `cross_dataset_protocol.ipynb`, so the intact run can be loaded from its saved seeds.

In [1]:
# 1.1 Settings: we change DATASET and run the notebook once per dataset
DATASET = 'sparkov'         # 'sparkov' or 'tabformer'
QUICK   = False             # True = 1 seed, 100 trees, 200 bootstrap resamples (a fast preview only)

SHUFFLE_SEED = 7            # the same shuffle for every model seed
SEEDS     = [42] if QUICK else [42, 43, 44, 45, 46]
N_TREES   = 100 if QUICK else 200
N_BOOT    = 200 if QUICK else 500
BOOT_SEED = 2026

# TabFormer: the same user sample and window as tabformer_validation.ipynb
TF_N_USERS, TF_SAMPLE_SEED, TF_WINDOW_START = 400, 42, '2015-01-01'

KAGGLE_SLUGS = {
    'sparkov':   'kartik2112/fraud-detection',
    'tabformer': 'ealtman2019/credit-card-transactions',
    'cc2026':    'mohankrishnathalla/credit-card-fraud-detection-dataset-2026',
}
assert DATASET in ('sparkov', 'tabformer'), 'the shuffle tests are for Sparkov and TabFormer'
print(f'dataset: {DATASET} | seeds: {SEEDS} | trees: {N_TREES} | bootstrap: {N_BOOT}')

dataset: sparkov | seeds: [42, 43, 44, 45, 46] | trees: 200 | bootstrap: 500


In [2]:
# 1.2 Download the dataset from Kaggle
import os, glob
from getpass import getpass

DATA_DIR = f'/content/data/{DATASET}'
if not glob.glob(f'{DATA_DIR}/**/*.csv', recursive=True):
    os.environ['KAGGLE_USERNAME'] = input('Kaggle username: ')
    os.environ['KAGGLE_KEY'] = getpass('Kaggle API key: ')
    !kaggle datasets download -d {KAGGLE_SLUGS[DATASET]} -p {DATA_DIR} --unzip
for f in sorted(glob.glob(f'{DATA_DIR}/**/*', recursive=True)):
    if os.path.isfile(f):
        print(f'{os.path.getsize(f) / 1e6:9.1f} MB  {f}')

Kaggle username: Subbareddy_subbz
Kaggle API key: ··········
Dataset URL: https://www.kaggle.com/datasets/kartik2112/fraud-detection
License(s): CC0-1.0
100% 202M/202M [00:02<00:00, 82.1MB/s]

    150.4 MB  /content/data/sparkov/fraudTest.csv
    351.2 MB  /content/data/sparkov/fraudTrain.csv


In [3]:
# 1.3 Clone the project repository to use our helpers
REPO = 'Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-'
if not os.path.exists('src/evaluation.py'):
    if not os.path.exists(REPO):
        github_token = getpass('Paste your GitHub token: ').strip()
        !git clone https://{github_token}@github.com/subbareddypullalacheruvu/{REPO}.git
    %cd {REPO}

Paste your GitHub token: ··········
Cloning into 'Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-'...
remote: Enumerating objects: 345, done.
remote: Counting objects: 100% (196/196), done.
remote: Compressing objects: 100% (152/152), done.
remote: Total 345 (delta 128), reused 53 (delta 44), pack-reused 149 (from 1)
Receiving objects: 100% (345/345), 11.94 MiB | 15.76 MiB/s, done.
Resolving deltas: 100% (206/206), done.
/content/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-


In [4]:
# 1.4 Imports and the Google Drive folders for progress and results
import sys
import numpy as np
from IPython.display import display
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

sys.path.append(os.getcwd())
from src.utils import fit_category_risk, apply_category_risk, haversine_distance
from src.evaluation import chronological_val_split, run_seeds, bootstrap_model, summarise, paired_difference, format_ci
pd.set_option('display.max_columns', 60)

from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/fraud_checkpoints/cross_dataset_protocol'
CHECKPOINT_DIR = f"{BASE}/{DATASET}{'_quick' if QUICK else ''}"
RESULTS_DIR = f'{BASE}/results'   # same folders as cross_dataset_protocol.ipynb
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print('progress ->', CHECKPOINT_DIR)
print('results  ->', RESULTS_DIR)

Mounted at /content/drive
progress -> /content/drive/MyDrive/fraud_checkpoints/cross_dataset_protocol/sparkov
results  -> /content/drive/MyDrive/fraud_checkpoints/cross_dataset_protocol/results


## 2. Loading the data
We use the same loaders as `cross_dataset_protocol.ipynb`, so the data and the helper columns are identical.

In [5]:
# 2.1 One loader per dataset
def csv_files():
    return glob.glob(f'{DATA_DIR}/**/*.csv', recursive=True)

def load_sparkov():
    files = csv_files()
    tr = pd.read_csv(next(f for f in files if f.endswith('fraudTrain.csv')))
    te = pd.read_csv(next(f for f in files if f.endswith('fraudTest.csv')))
    tr['_split'], te['_split'] = 'train', 'test'          # Kaggle's own train/test files, as in all our notebooks
    d = pd.concat([tr, te], ignore_index=True)
    d = d.drop(columns=[c for c in d.columns if c.startswith('Unnamed')])
    d['_time'] = pd.to_datetime(d['trans_date_trans_time'])
    d['_y'] = d['is_fraud'].astype(int)
    d['_card'] = d['cc_num'].astype(str)
    d['_amount'] = d['amt'].astype(float)
    d['_category'] = d['category'].astype(str)
    d['age'] = (d['_time'] - pd.to_datetime(d['dob'])).dt.days / 365.25
    d['gender_flag'] = (d['gender'] == 'M').astype(int)
    skip = {'trans_date_trans_time', 'cc_num', 'first', 'last', 'street', 'trans_num', 'unix_time',
            'dob', 'is_fraud', '_split'}
    native = [c for c in tr.columns if c not in skip and not c.startswith('Unnamed')]
    text = {'category', 'gender', 'city', 'state', 'job', 'merchant', 'zip'}
    return d, native, text

def load_tabformer():
    files = csv_files()
    users_csv = next(f for f in files if 'user' in os.path.basename(f).lower()
                     and 'transaction' not in os.path.basename(f).lower())
    txn_csv = max((f for f in files if 'transaction' in os.path.basename(f).lower()), key=os.path.getsize)
    users = pd.read_csv(users_csv)
    users['User'] = np.arange(len(users))               # transactions refer to users by row number
    rng = np.random.default_rng(TF_SAMPLE_SEED)
    sampled = set(rng.choice(users['User'].values, size=min(TF_N_USERS, len(users)), replace=False).tolist())
    parts = [ch[ch['User'].isin(sampled)] for ch in pd.read_csv(txn_csv, chunksize=2_000_000, low_memory=False)]
    raw = pd.concat(parts, ignore_index=True)
    txn_cols = list(raw.columns)

    d = raw.copy()
    d['_time'] = pd.to_datetime(dict(year=d['Year'], month=d['Month'], day=d['Day']), errors='coerce') \
                 + pd.to_timedelta(d['Time'].astype(str) + ':00', errors='coerce')
    d['_y'] = d['Is Fraud?'].astype(str).str.strip().str.lower().isin(['yes', '1', 'true']).astype(int)
    d['_amount'] = pd.to_numeric(d['Amount'].astype(str).str.replace(r'[$,]', '', regex=True), errors='coerce')
    d['_card'] = d['User'].astype(str) + '_' + d['Card'].astype(str)     # one card = user + card number
    d['_category'] = d['MCC'].astype(str)
    user_cols = [c for c in users.columns if c not in d.columns]
    d = d.merge(users[['User'] + user_cols], on='User', how='left')
    birth = pd.to_datetime(dict(year=d['Birth Year'], month=d['Birth Month'], day=1), errors='coerce')
    d['age'] = (d['_time'] - birth).dt.days / 365.25
    d['gender_flag'] = d['Gender'].astype(str).str.strip().str.upper().str.startswith('M').astype(int)
    d = d.dropna(subset=['_time'])

    skip = {'User', 'Card', 'Year', 'Month', 'Day', 'Time', 'Is Fraud?', 'Merchant Name', 'Zip',
            'Person', 'Address', 'Apartment', 'Zipcode', 'Birth Month'}
    native = [c for c in txn_cols + user_cols if c not in skip]
    text = {'MCC', 'Merchant State', 'Merchant City', 'State', 'Gender', 'City', 'Use Chip', 'Errors?'}
    return d, native, text

def load_cc2026():
    d = pd.read_csv(max(csv_files(), key=os.path.getsize), low_memory=False)
    raw_cols = list(d.columns)
    d['_time'] = pd.to_datetime(d['transaction_datetime'], errors='coerce')
    d['_y'] = pd.to_numeric(d['is_fraud'], errors='coerce').fillna(0).astype(int)
    d['_card'] = d['card_id'].astype(str)
    d['_amount'] = d['transaction_amount'].astype(float)
    d['_category'] = d['merchant_category'].astype(str)
    d['age'] = pd.to_numeric(d['customer_age'], errors='coerce')
    d['gender_flag'] = d['customer_gender'].astype(str).str.strip().str.upper().str.startswith('M').astype(int)
    d = d.dropna(subset=['_time'])
    # identifiers, the time stamp, the label and the two columns that describe the fraud itself
    skip = {'transaction_id', 'customer_id', 'card_id', 'merchant_id', 'transaction_datetime',
            'is_fraud', 'fraud_type', 'fraud_amount_lost'}
    native = [c for c in raw_cols if c not in skip]
    text = {'merchant_category', 'merchant_country', 'transaction_day_of_week', 'customer_gender',
            'customer_state', 'transaction_channel', 'device_type', 'browser', 'email_domain'}
    return d, native, text

d, NATIVE_COLS, TEXT_COLS = {'sparkov': load_sparkov, 'tabformer': load_tabformer}[DATASET]()
print(f'{len(d):,} rows, {d["_card"].nunique():,} cards, fraud rate {d["_y"].mean():.4%}')
print(f'{d["_time"].min()} to {d["_time"].max()}')
print(f'{len(NATIVE_COLS)} native columns:', NATIVE_COLS)

1,852,394 rows, 999 cards, fraud rate 0.5210%
2019-01-01 00:00:18 to 2020-12-31 23:59:34
13 native columns: ['merchant', 'category', 'amt', 'gender', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'merch_lat', 'merch_long']


## 3. Building the three versions of the data
`build(shuffle)` makes the same features as `cross_dataset_protocol.ipynb`, after shuffling either the merchant locations or the time gaps within each card. The rows stay in the same order in all three versions, so the test rows line up. The table at the end checks that each shuffle changes only its own part of the data.

In [6]:
# 3.1 Feature builder with an optional within-card shuffle
NIGHT_HOURS = [22, 23, 0, 1, 2, 3]
LOCATION_RAW = {'sparkov': ['merch_lat', 'merch_long'],
                'tabformer': ['Merchant City', 'Merchant State']}[DATASET]

base = d.sort_values(['_card', '_time']).reset_index(drop=True)
base['_gap_h'] = base.groupby('_card')['_time'].diff().dt.total_seconds() / 3600

def shuffle_within_card(df, cols, seed):
    # df is sorted by card; every card's values are permuted among that card's own rows
    rng = np.random.default_rng(seed)
    perm = df.assign(_k=rng.random(len(df))).sort_values(['_card', '_k'], kind='stable').index.values
    out = df.copy()
    out[cols] = df.loc[perm, cols].values        # same cards in the same order, so groups line up row for row
    assert (out['_card'].values == df.loc[perm, '_card'].values).all()
    return out

def build(shuffle=None):
    x = base
    if shuffle == 'location':
        x = shuffle_within_card(base, LOCATION_RAW, SHUFFLE_SEED)
    elif shuffle == 'time':
        x = shuffle_within_card(base, ['_gap_h'], SHUFFLE_SEED)
    else:
        x = base.copy()
    by_card = x.groupby('_card')

    if DATASET == 'sparkov':
        prev_lat, prev_lon = by_card['merch_lat'].shift(1), by_card['merch_long'].shift(1)
        x['_dist'] = haversine_distance(x['merch_lat'], x['merch_long'], prev_lat, prev_lon)
        x['_vel'] = x['_dist'] / (x['_gap_h'] + 1 / 3600)
        x['dist_home_log'] = np.log1p(haversine_distance(x['lat'], x['long'], x['merch_lat'], x['merch_long']))
        location = ['dist_home_log', 'dist_log']
    else:
        city = x['Merchant City'].astype(str)
        prev_city = by_card['Merchant City'].shift(1)
        x['_dist'] = np.where(prev_city.isna(), np.nan, (city != prev_city.astype(str)).astype(float))
        x['_vel'] = x['_dist'] / (x['_gap_h'] + 1 / 60)
        ms, has = x['Merchant State'], x['Merchant State'].notna()
        x['abroad'] = (has & (ms.astype(str).str.len() > 2)).astype(int)
        x['outside_home_state'] = (has & (ms.astype(str) != x['State'].astype(str))).astype(int)
        location = ['outside_home_state', 'abroad']
        x = x[x['_time'] >= TF_WINDOW_START].copy()

    x['gap_log'] = np.log1p(x['_gap_h']).fillna(-1)
    x['dist_log'] = np.log1p(x['_dist'].clip(lower=0)).fillna(-1)
    x['velocity_log'] = np.log1p(x['_vel'].clip(lower=0)).fillna(-1)
    x['amt_log'] = np.log1p(x['_amount'].clip(lower=0)).fillna(0)
    x['is_night'] = x['_time'].dt.hour.isin(NIGHT_HOURS).astype(int)
    x['age'] = x['age'].fillna(-1)
    x['day'] = x['_time'].dt.floor('D')
    return x.sort_values('_time').reset_index(drop=True), location

versions = {}
for name, shuffle in [('Intact', None), ('Location shuffled', 'location'), ('Time gap shuffled', 'time')]:
    versions[name], LOCATION = build(shuffle)
    print(f'{name}: {len(versions[name]):,} rows')
del base, d

# sanity checks: same rows and times in every version, and the shuffle really changed the values
ref = versions['Intact']
for name, x in versions.items():
    assert (x['_time'].values == ref['_time'].values).all() and (x['_y'].values == ref['_y'].values).all()
pd.DataFrame({name: {'gap_log changed %': (x['gap_log'] != ref['gap_log']).mean() * 100,
                     'location features changed %': (x[LOCATION] != ref[LOCATION]).any(axis=1).mean() * 100}
              for name, x in versions.items()}).round(1)

Intact: 1,852,394 rows
Location shuffled: 1,852,394 rows
Time gap shuffled: 1,852,394 rows


,Intact,Location shuffled,Time gap shuffled
gap_log changed %,0.0,0.0,99.9
location features changed %,0.0,100.0,0.0


## 4. Splitting by time and fitting `category_risk`
The split and `category_risk` are the same as in `cross_dataset_protocol.ipynb`. We check that the labels of the training and test rows are identical in the three versions.

In [7]:
# 4.1 Train / validation / test for each version
SIX = ['amt_log', 'category_risk', 'is_night', 'gap_log', 'age', 'gender_flag']
FEATURES = SIX + [c for c in LOCATION if c not in SIX]
print('Six + location:', FEATURES)

def split(x):
    if DATASET == 'sparkov':
        train, test = x[x['_split'] == 'train'], x[x['_split'] == 'test'].copy()
    else:
        cutoff_test = x['_time'].min() + 0.8 * (x['_time'].max() - x['_time'].min())
        train, test = x[x['_time'] < cutoff_test], x[x['_time'] >= cutoff_test].copy()
    train_sub, val, _ = chronological_val_split(train, '_time', val_frac=0.2)
    risk_map = fit_category_risk(train_sub, '_category', '_y')
    for p in (train_sub, val, test):
        p['category_risk'] = apply_category_risk(p, '_category', risk_map)
    return train_sub, val, test

splits = {name: split(x) for name, x in versions.items()}
del versions
train_sub, val, test = splits['Intact']
y_tr, y_val, y_test = train_sub['_y'], val['_y'], test['_y']
PREVALENCE = y_test.mean()
for name, (a, b, c) in splits.items():
    assert (c['_y'].values == y_test.values).all() and (a['_y'].values == y_tr.values).all()
print(f'test rows {len(y_test):,}, fraud {int(y_test.sum()):,}, rate {PREVALENCE:.4%}')

Six + location: ['amt_log', 'category_risk', 'is_night', 'gap_log', 'age', 'gender_flag', 'dist_home_log', 'dist_log']
test rows 555,719, fraud 2,145, rate 0.3860%


## 5. Training and measuring the drops
We train *Six + location* on each version with the Random Forest Tuned settings and five seeds. The intact run has the same name and Drive folder as in `cross_dataset_protocol.ipynb`, so its seeds are loaded if they already exist. All three runs are then scored on the same 500 resampled test days, which gives a 95% interval for each drop.

In [8]:
# 5.1 Train Six + location on each version (five seeds, threshold chosen on validation)
def make_rf(n_features):
    return lambda s: RandomForestClassifier(n_estimators=N_TREES, max_features=min(3, n_features),
                                            min_samples_leaf=5, random_state=s, n_jobs=-1)

RUN_NAMES = {'Intact': f'{DATASET} Six + location',                   # same name as cross_dataset_protocol.ipynb
             'Location shuffled': f'{DATASET} Six + location, location shuffled',
             'Time gap shuffled': f'{DATASET} Six + location, time gap shuffled'}
runs = {}
for name, (a, b, c) in splits.items():
    runs[name] = run_seeds(RUN_NAMES[name], make_rf(len(FEATURES)),
                           a[FEATURES], a['_y'], b[FEATURES], b['_y'], c[FEATURES], SEEDS,
                           checkpoint_dir=CHECKPOINT_DIR)

sparkov Six + location seed 42: loaded from checkpoint
sparkov Six + location seed 43: loaded from checkpoint
sparkov Six + location seed 44: loaded from checkpoint
sparkov Six + location seed 45: loaded from checkpoint
sparkov Six + location seed 46: loaded from checkpoint
sparkov Six + location, location shuffled seed 42: loaded from checkpoint
sparkov Six + location, location shuffled seed 43: loaded from checkpoint
sparkov Six + location, location shuffled seed 44: loaded from checkpoint
sparkov Six + location, location shuffled seed 45: loaded from checkpoint
sparkov Six + location, location shuffled seed 46: loaded from checkpoint
sparkov Six + location, time gap shuffled seed 42: loaded from checkpoint
sparkov Six + location, time gap shuffled seed 43: loaded from checkpoint
sparkov Six + location, time gap shuffled seed 44: loaded from checkpoint
sparkov Six + location, time gap shuffled seed 45: loaded from checkpoint
sparkov Six + location, time gap shuffled seed 46: loaded f

In [9]:
# 5.2 Bootstrap on the same resampled days, and the drop from the intact run
test_days = test['day'].values
boots, rows = {}, []
for name, seed_runs in runs.items():
    pairs = [(r['test_proba'], r['threshold']) for r in seed_runs]
    boots[name] = bootstrap_model(y_test, pairs, test_days, n_boot=N_BOOT, boot_seed=BOOT_SEED)
    s = summarise(y_test, pairs, boots[name])
    row = {'Dataset': DATASET, 'Run': name, 'Seeds': len(SEEDS), 'Prevalence': PREVALENCE,
           'Val PR-AUC': np.mean([r['val_pr_auc'] for r in seed_runs]),
           'PR-AUC': s['PR-AUC'], 'PR-AUC CI low': s['PR-AUC CI low'], 'PR-AUC CI high': s['PR-AUC CI high'],
           'Lift': s['PR-AUC'] / PREVALENCE,
           'F2': s['F2'], 'MCC': s['MCC'], 'Thresholds': s['Thresholds']}
    if name != 'Intact':
        dd = paired_difference(boots['Intact'], boots[name], 'PR-AUC')
        row.update({'Drop': dd['mean diff'], 'Drop CI low': dd['CI low'], 'Drop CI high': dd['CI high'],
                    'P(intact > shuffled)': dd['P(A > B)']})
    rows.append(row)
    print(f'{name}: done')

result = pd.DataFrame(rows)
out = f"{RESULTS_DIR}/point2_{DATASET}{'_quick' if QUICK else ''}.csv"
result.to_csv(out, index=False)
os.makedirs('results', exist_ok=True)
result.to_csv(f"results/point2_{DATASET}{'_quick' if QUICK else ''}.csv", index=False)
print('saved ->', out)

pd.DataFrame({
    'PR-AUC [95% CI]': [f"{r['PR-AUC']:.4f} [{r['PR-AUC CI low']:.4f}, {r['PR-AUC CI high']:.4f}]" for _, r in result.iterrows()],
    'Lift': result['Lift'].round(1).values,
    'Drop from intact [95% CI]': ['' if r['Run'] == 'Intact' else
                                  f"{r['Drop']:.4f} [{r['Drop CI low']:.4f}, {r['Drop CI high']:.4f}]" for _, r in result.iterrows()],
}, index=result['Run'])

Intact: done
Location shuffled: done
Time gap shuffled: done
saved -> /content/drive/MyDrive/fraud_checkpoints/cross_dataset_protocol/results/point2_sparkov.csv


,PR-AUC [95% CI],Lift,Drop from intact [95% CI]
Run,,,
Intact,"0.8327 [0.8123, 0.8511]",215.7,
Location shuffled,"0.8344 [0.8140, 0.8523]",216.2,"-0.0017 [-0.0040, 0.0005]"
Time gap shuffled,"0.8096 [0.7886, 0.8283]",209.7,"0.0230 [0.0163, 0.0301]"


## 6. Table 2: the shuffle tests on both datasets
We read the saved results of every finished dataset. Each cell is the drop in test PR-AUC from the intact model, with its 95% interval and the drop as a share of the intact PR-AUC. A drop whose interval is above zero means that the shuffled part of the data carries signal.

In [10]:
# 6.1 Combine Sparkov and TabFormer
NAMES = {'sparkov': 'Sparkov', 'tabformer': 'TabFormer'}
files = sorted(glob.glob(f'{RESULTS_DIR}/point2_*.csv'))
print('found:', [os.path.basename(f) for f in files])
allres = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
allres = (allres.sort_values('Seeds', ascending=False)
                .drop_duplicates(['Dataset', 'Run']).reset_index(drop=True))

intact = allres[allres['Run'] == 'Intact'].set_index('Dataset')['PR-AUC']
table2 = {}
for ds in [x for x in ['sparkov', 'tabformer'] if x in set(allres['Dataset'])]:
    r = allres[allres['Dataset'] == ds].set_index('Run')
    table2[NAMES[ds]] = {'Intact PR-AUC': f"{r.loc['Intact', 'PR-AUC']:.3f} [{r.loc['Intact', 'PR-AUC CI low']:.3f}, {r.loc['Intact', 'PR-AUC CI high']:.3f}]"}
    for run in ['Location shuffled', 'Time gap shuffled']:
        x = r.loc[run]
        table2[NAMES[ds]][f'{run}: drop'] = (f"{x['Drop']:.3f} [{x['Drop CI low']:.3f}, {x['Drop CI high']:.3f}] "
                                             f"({x['Drop'] / intact[ds]:.0%})")
table2 = pd.DataFrame(table2).T
table2.to_csv(f'{RESULTS_DIR}/table2_shuffle_tests.csv')
print('Table 2: drop in test PR-AUC when one part of the data is shuffled within each card')
table2

found: ['point2_sparkov.csv']
Table 2: drop in test PR-AUC when one part of the data is shuffled within each card


,Intact PR-AUC,Location shuffled: drop,Time gap shuffled: drop
Sparkov,"0.833 [0.812, 0.851]","-0.002 [-0.004, 0.001] (-0%)","0.023 [0.016, 0.030] (3%)"


## 7. Discussion

### Sparkov

**The shuffles did what we intended.** The time-gap shuffle changed `gap_log` in 99.9% of the rows and left every location feature unchanged. The location shuffle changed the location features in every row and left `gap_log` unchanged. The test set is the same as in all our Sparkov notebooks: 555,719 transactions with 2,145 fraud (0.386%).

**The intact model.** *Six + location* scores a test PR-AUC of 0.833 [0.812, 0.851], a lift of 216 over the fraud rate. This is slightly below our six features alone (0.844 in `final_model_gap.ipynb`), so the two distance features do not help on Sparkov.

**Shuffling the locations changes nothing.** With the merchant locations shuffled within each card, the PR-AUC is 0.834 [0.814, 0.852]. The drop is −0.002 [−0.004, 0.001], and the interval includes zero, so we cannot tell the two models apart. The validation PR-AUC also stays at about 0.874. Where the merchant is carries no fraud signal in Sparkov, which agrees with our earlier test in `velocity_breakdown.ipynb`, where shuffled locations gave 0.833.

**Shuffling the time gaps lowers the PR-AUC.** With the time gaps shuffled within each card, the PR-AUC falls to 0.810 [0.789, 0.828]. The drop is 0.023 [0.016, 0.030], and the whole interval is above zero. The validation PR-AUC falls in the same way, from about 0.874 to about 0.855 in every seed.

**Why the drop is about 3%.** Most of the signal in Sparkov comes from the amount, `category_risk` and the time of day, which the shuffle does not touch. The time gap adds the rest: without it, our features score 0.817, and with it they score 0.844, a gain of about 0.027 (`velocity_breakdown.ipynb`). The time shuffle removes 0.023, which is close to that whole gain. The shuffled model (0.810) even scores slightly below the model without any gap (0.817), because a shuffled gap is noise that the trees still try to use. So the shuffle removes the contribution of the time gap almost completely, and the location shuffle removes nothing.

**What this means.** On Sparkov the signal is temporal, not spatial: breaking the link between each transaction and its time gap hurts the model, while breaking the link with its location does not. This is the first half of the shuffle test. The TabFormer run will show whether the opposite holds there.